# Neutron rates

A deuterium beam in a deuterium plasma makes neutrons by D(d,n)³He in two ways: the fast beam ions fuse with the thermal ions while they slow down (beam–target), and the thermal ions fuse with each other (thermonuclear). `neutron_rates` gives both, shell by shell.

A fast ion born with speed $v_0$ slows down by $\mathrm{d}v/\mathrm{d}t = -(a_e v + a_i/v^2)$, the classical drag of `slow_down`, and on the way reacts with the $n_t$ target ions per m³ at the rate $n_t \langle\sigma v_\mathrm{rel}\rangle_b$. Over its slowing down it makes

$$
N = n_t \int_{v_\mathrm{end}}^{v_0} \frac{\langle\sigma v_\mathrm{rel}\rangle_b(v)}{a_e v + a_i/v^2}\,\mathrm{d}v
$$

reactions, where $\langle\sigma v_\mathrm{rel}\rangle_b$ is the Bosch–Hale cross section (Nucl. Fusion 32, 611, 1992, in `plasmacoll.fusion`) averaged over a Maxwellian target at $T_i$. A fast ion counts until it has slowed down to $\tfrac32 T_i$; from there on it is a thermal ion. The thermonuclear rate is $\tfrac12 n_D^2 \langle\sigma v\rangle(T_i)$.

In [ ]:
import numpy as np
import plasma_plots
import xarray as xr

import freeofcharge as foc
from freeofcharge.neutrons import neutron_rates

plasma_plots.set_backend("plotly")  # interactive figures, embedded in the docs
plasma_plots.set_math("latex")  # formulas in the figures typeset as LaTeX

equilibrium = foc.Equilibrium.solovev(6.2, 1.8, 1.7, 5.3, wall_psi_n=1.3)
rho_edges = np.linspace(0.0, 1.0, 21)
surfaces = equilibrium.flux_surfaces(rho_edges)
origin, direction = foc.tangential_injection(5.3, 12.0)


def run(energy=1.0e6, power=16.5e6, temperature=2e4, ne=1e20):
    plasma = foc.Plasma.parabolic(
        ne, temperature, ions={"D": (2, 1, 0.96), "He": (4, 2, 0.02)}
    )
    beam = foc.NeutralBeam.gaussian(
        origin,
        direction,
        energy,
        power,
        width=0.3,
        divergence=0.005,
        markers=300,
        seed=1,
    )
    deposition = foc.deposit(
        beam, equilibrium, plasma, step=0.03, rho_edges=rho_edges, seed=2
    )
    return neutron_rates(deposition, plasma, surfaces), deposition


rates, deposition = run()
print(f"absorbed beam power {deposition.absorbed_power / 1e6:.1f} MW")
print(f"beam-target  {rates.total_beam_target:.2e} n/s")
print(f"thermonuclear {rates.total_thermonuclear:.2e} n/s")
print(f"neutrons per 1 MeV deuteron: {rates.reactions_per_ion:.1e}")

This is the tokamak of the heating and current drive example: 6.2 m, 5.3 T, $n_e = 10^{20}$ m⁻³ and $T_e = T_i = 20$ keV in the centre, with a 16.5 MW, 1 MeV deuterium beam. At 20 keV the thermonuclear neutrons outnumber the beam–target ones about eight to one; about one 1 MeV deuteron in 350 makes a neutron before it is thermal.

## Emissivity profile

The reactions are counted on the shell where the fast ion is born (zero orbit width), as the heating of `slow_down` is. The beam–target emissivity follows the births, the thermonuclear one $n_D^2 \langle\sigma v\rangle(T_i)$, which falls steeply with the temperature towards the edge:

In [ ]:
profiles = rates.to_dataset()
emissivity = (
    profiles[["beam_target_emissivity", "thermonuclear_emissivity"]]
    .rename(
        {
            "beam_target_emissivity": "beam-target",
            "thermonuclear_emissivity": "thermonuclear",
        }
    )
    .to_array("source", name="neutron emissivity")
)
emissivity.attrs["units"] = "1/(m^3 s)"
emissivity.plasma.plot.profiles(x="rho", over="source")

## Beam–target and thermonuclear against the temperature

Towards lower temperature the thermonuclear rate falls with $\langle\sigma v\rangle(T_i)$, much faster than the beam–target rate, which falls with the slowing-down time $\tau_s \propto T_e^{3/2}$, the time a fast ion has to react. In this plasma the two are equal at about 5 keV in the centre, and below it the beam–target neutrons dominate, as in present deuterium tokamaks heated by neutral beams:

In [ ]:
temperatures = np.array([3e3, 5e3, 8e3, 12e3, 16e3, 20e3, 25e3])
scan = [run(temperature=t)[0] for t in temperatures]
versus_t = xr.Dataset(
    {
        "beam-target": ("T", [r.total_beam_target for r in scan]),
        "thermonuclear": ("T", [r.total_thermonuclear for r in scan]),
    },
    coords={
        "T": (
            "T",
            temperatures / 1e3,
            {"long_name": "central $T_e = T_i$", "units": "keV"},
        )
    },
).to_array("source", name="neutron rate")
versus_t.attrs["units"] = "1/s"
for t, r in zip(temperatures, scan, strict=True):
    print(
        f"T = {t / 1e3:4.0f} keV: beam-target {r.total_beam_target:.2e}, "
        f"thermonuclear {r.total_thermonuclear:.2e} n/s"
    )
versus_t.plasma.plot.profiles(x="T", over="source", logy=True)

## Rate against the beam energy

At a fixed beam power a lower energy means more ions, each with far fewer reactions: the D–D cross section rises steeply up to a few hundred keV, and a faster ion also slows down for longer. Per megawatt absorbed, the beam–target rate at 20 keV grows some thirtyfold from 80 keV to 1 MeV:

In [ ]:
energies = np.array([80e3, 150e3, 300e3, 500e3, 1000e3])
per_mw = []
for energy in energies:
    r, d = run(energy=energy)
    per_mw.append(r.total_beam_target / (d.absorbed_power / 1e6))
    print(
        f"{energy / 1e3:5.0f} keV: {per_mw[-1]:.2e} n/s per MW absorbed, "
        f"{r.reactions_per_ion:.1e} per ion"
    )
versus_e = xr.DataArray(
    per_mw,
    coords={"E": ("E", energies / 1e3, {"long_name": "beam energy", "units": "keV"})},
    dims="E",
    name="beam-target neutrons per MW",
    attrs={"units": "1/(s MW)"},
)
versus_e.plasma.plot.lineout(x="E", logy=True)

## Order of magnitude: a JET-like plasma

JET deuterium plasmas heated by 10–20 MW of 80–140 keV deuterium beams reach total D–D neutron rates of $10^{15}$ to $10^{16}$ n/s, most of them beam–target. The same model in a JET-sized Solovev equilibrium (2.96 m, 1 m, elongation 1.6, 3 T) with 10 MW of 110 keV deuterium at $n_e = 5\times10^{19}$ m⁻³ and 6 keV gives a rate in that range:

In [ ]:
jet = foc.Equilibrium.solovev(2.96, 1.0, 1.6, 3.0, wall_psi_n=1.3)
jet_edges = np.linspace(0.0, 1.0, 11)
jet_plasma = foc.Plasma.parabolic(5e19, 6e3, ions={"D": (2, 1, 1.0)})
jet_beam = foc.NeutralBeam.gaussian(
    *foc.tangential_injection(2.96, 6.0),
    110e3,
    10e6,
    width=0.15,
    divergence=0.01,
    markers=300,
    seed=1,
)
jet_deposition = foc.deposit(
    jet_beam, jet, jet_plasma, step=0.02, rho_edges=jet_edges, seed=2
)
jet_rates = neutron_rates(jet_deposition, jet_plasma, jet.flux_surfaces(jet_edges))
print(
    f"beam-target {jet_rates.total_beam_target:.1e} n/s, thermonuclear {jet_rates.total_thermonuclear:.1e} n/s"
)
assert 1e15 < jet_rates.total_beam_target < 2e16

What the model leaves out: the orbit width and the losses of the fast ions (each reacts on its birth shell), energy diffusion during the slowing down, beam–beam reactions and charge exchange. `maxwellian_target=False` takes the target at rest, the usual approximation for beam ions much faster than the thermal ions; at 1 MeV it changes the rate by a few percent, at 100 keV and 5 keV by about 40%. For D–T, pass `reaction="T(d,n)4He"` with a tritium species in the plasma.